#RAG: Retrieval-Augumented-Generation(RAG)

RAG is an enhanced technique that enables large language model accuracy and relevance by fetching external up-to data before responsing . which fine tuning lacks . earlier whenever new data was update everytime we need to fine tune the model for the correct response but with rag it became easy

think of it like we have pre exiting data from where we want the answer so we just retrive/fetch the database and provide it to the LLM and then ask the question

So we are sending the question but also send the context of database along with it
LLM is great at answering the question but it doesnot have our data so we basically fetching the data from database and sending to LLM along with the Question

In [103]:
!pip install -q -U langchain-community langchain-core langchain-google-genai langchainhub chromadb sentence-transformers

In [104]:
!pip install langchain_groq

In [105]:
from langchain_groq import ChatGroq

In [106]:
from google.colab import userdata
import os
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
except:
    # Fallback if you want to paste it directly (not recommended for sharing)
    os.environ["GROQ_API_KEY"] = input("Enter your Groq API Key: ")

In [107]:
print("Loading data...")

Loading data...


Now i am webscrapping my web page that why i am using webbase loader


In [108]:
from langchain_community.document_loaders import WebBaseLoader


# Load the data
loader = WebBaseLoader("https://cloud.google.com/use-cases/retrieval-augmented-generation")
docs = loader.load()

Now i have entire page content
so i am splitting the docs content into some chunnnks and save into vectore database

For that we used
#RecursiveCharacterTextSplitter

In [109]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [110]:
print("Splitting text...")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

Splitting text...


In [111]:
print(f"Successfully split documents into {len(splits)} chunks.")

Successfully split documents into 91 chunks.


printing the slipts which will contain 1000 words with some overlaps

In [112]:
print(splits[0])
print(splits[1])

page_content='What is Retrieval-Augmented Generation (RAG)? | Google CloudPage ContentsTopicsRAGWhat is Retrieval-Augmented Generation (RAG)?RAG (Retrieval-Augmented Generation) is an AI framework that combines the strengths of traditional information retrieval systems (such as search and databases) with the capabilities of generative large language models (LLMs). By combining your data and world knowledge with LLM language skills, grounded generation is more accurate, up-to-date, and relevant to your' metadata={'source': 'https://cloud.google.com/use-cases/retrieval-augmented-generation', 'title': 'What is Retrieval-Augmented Generation (RAG)? | Google Cloud', 'description': 'Retrieval-augmented generation (RAG) combines LLMs with external knowledge bases to improve their outputs. Learn more with Google Cloud.', 'language': 'en-US'}
page_content='more accurate, up-to-date, and relevant to your specific needs. Check out this ebook to unlock your “Enterprise Truth.”Get started for free3

Total Number of splits

In [113]:
print(len(splits))

91


In [114]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [115]:
print("Embedding data (this may take a moment)...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Embedding data (this may take a moment)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#Adding Documents to vector database

using Chroma DB
and OpenAIEmbbeing to convets splits into Embeeding

In [116]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)
vectorstore.persist()

In [117]:

print(vectorstore._collection.count())
print(vectorstore._collection.get())


455
{'ids': ['51ff8b65-1296-424b-bcea-6d32c3584567', '7c036f71-6a60-4321-b706-172f8918b3a0', '04796061-48ef-4d84-bec2-5bff41f00ce2', 'f40c9da7-9cec-44f9-8fd3-3e607e1331d4', '731965ec-b858-4a74-990a-2676ce582ab0', '9ebd1a96-271a-4bc0-916c-60ed635ecd87', 'a332241d-e571-4495-ba20-85a0e4808695', '101d8b86-0f99-4ff7-9b81-dbbfd834a4fc', '7301f229-57ac-434d-beed-2d6887479031', '3e3f7f7b-922f-4905-a397-f58bc0c45b0e', '85125fa7-0455-4a04-a276-49ae2bcc6be9', 'bd8694b1-46a8-4c8c-9978-0a705c32512e', '5ad5382d-1f61-4471-b5b2-237bd805ef88', 'f1f3962f-7942-4d03-8d36-8ca54f633b61', '22de36c1-597b-4752-9db9-fc5649a7d293', 'a2cddac1-e5a4-4fc4-ab5f-84c2d536470c', '5e128b1e-ad24-4501-b461-c0dca6cbf826', 'bc8f4c04-390a-4595-9b2d-d5d7386f986d', 'e19c5f18-32d9-43cf-907f-9299dcb3f3e1', '11b425da-404f-4c03-9b54-41b999eb9c90', '21e81862-b8e3-4e74-b659-98639e59b681', '99f678c3-f5b2-4533-b1e3-5c9953436ebc', '9deb961a-4742-43f0-b706-db7d31775470', '36c5ecb7-8b28-40fa-a015-a1f88eeb47e3', '1d1a270f-a3f9-4f32-b546-78

#CREATING A RAG PIPELINE

In [118]:
!pip install -U huggingface-hub

it is an ai framework that enables LLM by fetching relevent, external data to ground responses in facts. it operates by taking user query, retrieving content form the knowledge base and using the information to generate accurate answers.

In [119]:
retriever = vectorstore.as_retriever()


In [120]:
!pip install langchainhub

In [121]:
from langchain_core.prompts import ChatPromptTemplate


template = """
You are an expert technical documentation assistant.

Your job is to answer the user's question using ONLY the provided context.
Follow these rules strictly:

1. If the answer is found in the context, provide a clear and structured explanation.
2. If the answer is NOT found in the context, say:
   "I don't have enough information in the provided documentation to answer this."
3. Do NOT make up information.
4. Keep explanations concise but clear.
5. When helpful, include code snippets from the context.
6. If the question is conceptual, explain step-by-step.

---------------------


User Question:
{question}

Answer:{context}
"""

prompt = ChatPromptTemplate.from_template(template)

#setting up LLM


In [122]:
!pip install -U transformers accelerate


In [123]:
#import google.generativeai as genai

In [129]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGroq(model_name="openai/gpt-oss-safeguard-20b", temperature=0)

In [130]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [131]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [132]:
print("\n--- RAG SYSTEM READY ---\n")


--- RAG SYSTEM READY ---



In [134]:
response = rag_chain.invoke("What is Rag?")
print(response)


**RAG** (Retrieval‑Augmented Generation) is a method that combines a semantic search‑based retrieval step with a large language model (LLM) to produce text that is grounded in external knowledge.  

Key points from the documentation:

- **Retrieval mechanism**: RAG relies on a curated knowledge base and a high‑quality semantic search to pull in the most relevant information for a given query or context.  
- **Generation**: The LLM is fine‑tuned or prompt‑engineered to generate responses *solely* from the retrieved content, which reduces off‑topic or contradictory output.  

In short, RAG is a pipeline that first retrieves relevant documents and then uses those documents to guide the LLM’s generation.


In [135]:
response = rag_chain.invoke("what is retrieval?")
print(response)

**Retrieval** in the context of Retrieval‑Augmented Generation (RAG) refers to the use of traditional information‑retrieval systems—such as search engines or database queries—to fetch relevant documents or data. These retrieved pieces of information are then combined with a generative large language model (LLM) to produce grounded, up‑to‑date, and context‑aware responses.


In [136]:
response = rag_chain.invoke("How does RAG help reduce hallucinations in LLMs?")
print(response)

RAG reduces hallucinations by grounding the model’s output in real, up‑to‑date documents rather than relying solely on its internal knowledge.  

* **Factual grounding** – The retrieved passages supply concrete evidence that the LLM can cite, limiting the chance of fabricating facts.  
* **Access to fresh information** – Because the model can pull in recent data, it is less likely to generate outdated or invented details.  

Thus, by tying responses to retrieved content, RAG keeps the model’s answers anchored to verifiable sources and mitigates hallucinated statements.
